<a href="https://colab.research.google.com/github/JinVibe/pytorch-example/blob/fix-mnist/AI_10team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### 인공지능 프로젝트 폴더 구조 만들기
import os

# 최상위 폴더
base_dir = "/content/wildfire_project/data"

# 데이터셋 분할 종류와 클래스 이름들
splits = ["train", "val", "test"]
categories = ["fire", "smoke", "normal"]

# 폴더 생성
for split in splits:
    for category in categories:
        dir_path = os.path.join(base_dir, split, category)
        os.makedirs(dir_path, exist_ok=True)
        print(f"폴더 생성됨: {dir_path}")


폴더 생성됨: /content/wildfire_project/data/train/fire
폴더 생성됨: /content/wildfire_project/data/train/smoke
폴더 생성됨: /content/wildfire_project/data/train/normal
폴더 생성됨: /content/wildfire_project/data/val/fire
폴더 생성됨: /content/wildfire_project/data/val/smoke
폴더 생성됨: /content/wildfire_project/data/val/normal
폴더 생성됨: /content/wildfire_project/data/test/fire
폴더 생성됨: /content/wildfire_project/data/test/smoke
폴더 생성됨: /content/wildfire_project/data/test/normal


In [ ]:
### 이미지 데이터셋 input.
## Forest Fire Dataset - https://www.kaggle.com/datasets/alik05/forest-fire-dataset
## fire / nofire
import kagglehub

# Download latest version
path_fire_nofire_dataset = kagglehub.dataset_download("alik05/forest-fire-dataset")

print("Path to dataset files:", path_fire_nofire_dataset) #/kaggle/input/forest-fire-dataset


Path to dataset files: /kaggle/input/forest-fire-dataset


In [ ]:
### 이미지 데이터셋 input
# Smoke Fire Detection YOLO - https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo
# smoke / fire

# 이 dataset은 우리가 분류를 해줘야한다
# txt 파일의 classes에서 '0: Smoke', '1: Fire'

# Only fire	1,164 -> fire로 분류
# Only smoke	5,867 -> smoke로 분류
# Fire and smoke	4,658 -> fire로 분류
# None	9,838 -> txt 공백이면 txt와 jpg 파일 모두 삭제하기

import kagglehub

# Download latest version
path_smoke_fire_dataset = kagglehub.dataset_download("sayedgamal99/smoke-fire-detection-yolo")

print("Path to dataset files:", path_smoke_fire_dataset) # /kaggle/input/smoke-fire-detection-yolo

Path to dataset files: /kaggle/input/smoke-fire-detection-yolo


In [ ]:
# /kaggle/input/... kaggle 구조의 읽기 전용이므로 데이터를 수정, 이동시킬 수 없음.
# /content/wildfire_project/data/ colab 구조로 복사해서 가공해야 함.

In [ ]:
# /kaggle/input/forest-fire-dataset/Forest Fire Dataset/
# ├── Testing/
# └── Training/
#     ├── fire/
#     └── nofire/

# 현재 이렇게 이미 Training과 Testing이 나뉘어져 있으므로
# 이 디렉토리 구조 기반으로 /content/wildfire_project/data/으로 복사한다.

# 사람이 의도적으로 Training와 Testing으로 나눈 데이터셋이라면,
# 그대로 쓰는 게 데이터 편향을 줄이고 신뢰성 있는 실험을 만들 수 있다고 함!

# /kaggle/input/... 이곳에서 val 폴더가 따로 없으므로
# Training에서 일부를 분할해서 /content/...의 val 파일로 넣어주겠음. (8:2)

In [ ]:
## fire
# 원본 경로: /kaggle/input/forest-fire-dataset/Forest Fire Dataset/Training/fire/
# 복사할 train 경로: /content/wildfire_project/data/train/fire/
# 복사할 val 경로: /content/wildfire_project/data/val/fire/

## normal
# 원본 경로: /kaggle/input/forest-fire-dataset/Forest Fire Dataset/Training/nofire/
# 복사할 train 경로: /content/wildfire_project/data/train/normal/
# 복사할 val 경로: /content/wildfire_project/data/val/normal/

In [ ]:
# ## 중복 복사 방지를 위해 폴더 내용 비우기 - 비워졌는지 확인용
# import os

# # 대상 경로
# train_fire = "/content/wildfire_project/data/train/fire/"
# val_fire = "/content/wildfire_project/data/val/fire/"
# train_normal = "/content/wildfire_project/data/train/normal/"
# val_normal = "/content/wildfire_project/data/val/normal/"

# # 폴더 내용 비우기
# def clear_folder(folder):
#     for file in os.listdir(folder):
#         file_path = os.path.join(folder, file)
#         if os.path.isfile(file_path):
#             os.remove(file_path)

# # 폴더 생성 (없으면 생성, 있으면 유지) + 비우기
# for path in [train_fire, val_fire, train_normal, val_normal]:
#     os.makedirs(path, exist_ok=True)
#     clear_folder(path)

# # 실제로 비워졌는지 확인
# for path in [train_fire, val_fire, train_normal, val_normal]:
#     file_count = len(os.listdir(path))
#     print(f"{path} → 이미지 개수: {file_count}")


In [ ]:
import os
import shutil
from glob import glob
import random

# 원본 경로
src_fire = "/kaggle/input/forest-fire-dataset/Forest Fire Dataset/Training/fire/"
src_normal = "/kaggle/input/forest-fire-dataset/Forest Fire Dataset/Training/nofire/"

# 대상 경로
train_fire = "/content/wildfire_project/data/train/fire/"
val_fire = "/content/wildfire_project/data/val/fire/"
train_normal = "/content/wildfire_project/data/train/normal/"
val_normal = "/content/wildfire_project/data/val/normal/"

# 폴더 비우기
def clear_folder(folder):
    for file in os.listdir(folder):
        file_path = os.path.join(folder, file)
        if os.path.isfile(file_path):
            os.remove(file_path)

# 폴더 생성 (없으면 생성, 있으면 유지)
for path in [train_fire, val_fire, train_normal, val_normal]:
    os.makedirs(path, exist_ok=True)
    clear_folder(path)

# 이미지 분할 함수
def split_data(src_dir, train_dir, val_dir, val_ratio=0.2, class_name=""):
    image_paths = glob(os.path.join(src_dir, "*"))
    random.shuffle(image_paths)

    val_count = int(len(image_paths) * val_ratio)
    val_images = image_paths[:val_count]
    train_images = image_paths[val_count:]

    for image_path in train_images:
        shutil.copy(image_path, os.path.join(train_dir, os.path.basename(image_path)))

    for image_path in val_images:
        shutil.copy(image_path, os.path.join(val_dir, os.path.basename(image_path)))

    print(f"[{class_name}] 총 이미지: {len(image_paths)} | Train: {len(train_images)} | Val: {len(val_images)}")

# 실행
split_data(src_fire, train_fire, val_fire, class_name="fire")
split_data(src_normal, train_normal, val_normal, class_name="normal")


[fire] 총 이미지: 760 | Train: 608 | Val: 152
[normal] 총 이미지: 760 | Train: 608 | Val: 152


In [ ]:
# 일단 여기까지 fire, nofire 복사 완료
# smoke와 fire 데이터셋은 txt 파일의 class 번호를 보고
# non을 삭제한 다음에 smoke와 fire을 구분해줘야함.

In [ ]:
import os
import shutil

# 클래스 번호 정의
CLASS_MAP = {0: "smoke", 1: "fire"}

# 데이터셋 경로
base_src = "/kaggle/input/smoke-fire-detection-yolo/data"
base_dst = "/kaggle/working/data"

# Split 목록
splits = ["train", "val", "test"]

# 디렉토리 생성
for split in splits:
    for cls in ["fire", "smoke"]:
        os.makedirs(os.path.join(base_dst, split, cls), exist_ok=True)

# 빈 라벨 파일 카운트용 변수
empty_label_count = 0

# 각 split에 대해 처리
for split in splits:
    labels_path = os.path.join(base_src, split, "labels")
    images_path = os.path.join(base_src, split, "images")

    label_files = [f for f in os.listdir(labels_path) if f.endswith(".txt")]

    for label_file in label_files:
        label_full_path = os.path.join(labels_path, label_file)

        # 라벨 읽기
        with open(label_full_path, "r") as f:
            content = f.read().strip()

        # 내용이 없을 경우만 출력
        if not content:
            print(f"빈 라벨 파일: {label_file} | 내용:\n{content}\n")
            empty_label_count += 1
            continue

        # 클래스 번호만 추출
        lines = content.splitlines()
        class_ids = [int(line.strip().split()[0]) for line in lines if line.strip()]

        # fire가 하나라도 있으면 fire로 분류, 없으면 smoke
        target_class = "fire" if 1 in class_ids else "smoke"

        # 이미지 파일 경로
        image_filename = label_file.replace(".txt", ".jpg")
        src_image_path = os.path.join(images_path, image_filename)
        dst_image_path = os.path.join(base_dst, split, target_class, image_filename)

        if os.path.exists(src_image_path):
            shutil.copy(src_image_path, dst_image_path)
        else:
            print(f"이미지 파일 없음: {image_filename} (라벨 파일: {label_file})")

# 빈 라벨 파일 총 개수 출력
print(f"\n총 빈 라벨 파일 수: {empty_label_count}")


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
빈 라벨 파일: WEB10042.txt | 내용:


빈 라벨 파일: WEB10352.txt | 내용:


빈 라벨 파일: AoF07689.txt | 내용:


빈 라벨 파일: AoF06919.txt | 내용:


빈 라벨 파일: AoF07702.txt | 내용:


빈 라벨 파일: AoF07493.txt | 내용:


빈 라벨 파일: WEB10422.txt | 내용:


빈 라벨 파일: AoF06760.txt | 내용:


빈 라벨 파일: AoF07216.txt | 내용:


빈 라벨 파일: AoF07430.txt | 내용:


빈 라벨 파일: WEB09556.txt | 내용:


빈 라벨 파일: WEB09448.txt | 내용:


빈 라벨 파일: WEB09550.txt | 내용:


빈 라벨 파일: AoF07423.txt | 내용:


빈 라벨 파일: AoF07282.txt | 내용:


빈 라벨 파일: WEB09999.txt | 내용:


빈 라벨 파일: PublicDataset01316.txt | 내용:


빈 라벨 파일: WEB10410.txt | 내용:


빈 라벨 파일: AoF07632.txt | 내용:


빈 라벨 파일: WEB10174.txt | 내용:


빈 라벨 파일: WEB10073.txt | 내용:


빈 라벨 파일: WEB10186.txt | 내용:


빈 라벨 파일: PublicDataset01299.txt | 내용:


빈 라벨 파일: AoF06924.txt | 내용:


빈 라벨 파일: AoF07713.txt | 내용:


빈 라벨 파일: AoF06925.txt | 내용:


빈 라벨 파일: WEB09908.txt | 내용:


빈 라벨 파일: AoF06844.txt | 내용:


빈 라벨 파일: WEB10359.txt | 내용:


빈 라벨 파일: AoF07210.txt | 내용:


빈 라벨 파일: WEB09642.txt | 내용:


빈 라벨 파일: AoF07

In [ ]:
## kaggle Dataset Statistics와 비교

import os

# 기준 경로
base_dir = "/kaggle/working/data"

# 클래스별 총합을 저장할 딕셔너리
total_counts = {"fire": 0, "smoke": 0}

# split과 class 조합 순회
for split in ["train", "val", "test"]:
    for cls in ["fire", "smoke"]:
        dir_path = os.path.join(base_dir, split, cls)
        if os.path.exists(dir_path):
            count = len([f for f in os.listdir(dir_path) if f.lower().endswith((".jpg"))])
            print(f"{dir_path} → {count}개")
            total_counts[cls] += count
        else:
            print(f"폴더 없음: {dir_path}")

# 클래스별 총합 출력
print("\nfire 총합:", total_counts["fire"])
print("smoke 총합:", total_counts["smoke"])


/kaggle/working/data/train/fire → 3828개
/kaggle/working/data/train/smoke → 3836개
/kaggle/working/data/val/fire → 879개
/kaggle/working/data/val/smoke → 845개
/kaggle/working/data/test/fire → 1115개
/kaggle/working/data/test/smoke → 1186개

fire 총합: 5822
smoke 총합: 5867


In [ ]:
## kaggle/working/... 에서 content/...로 복사하기

import os
import shutil

# 원본 및 대상 루트 디렉토리
src_root = "/kaggle/working/data"
dst_root = "/content/wildfire_project/data"

# split과 클래스 조합
splits = ["train", "val", "test"]
classes = ["fire", "smoke"]

# 디렉토리 생성 및 복사
for split in splits:
    for cls in classes:
        src_dir = os.path.join(src_root, split, cls)
        dst_dir = os.path.join(dst_root, split, cls)

        # 대상 디렉토리 생성
        os.makedirs(dst_dir, exist_ok=True)

        # 이미지 복사
        for file in os.listdir(src_dir):
            if file.lower().endswith((".jpg")):
                shutil.copy(os.path.join(src_dir, file), os.path.join(dst_dir, file))

        print(f"복사 완료: {src_dir} → {dst_dir}")


복사 완료: /kaggle/working/data/train/fire → /content/wildfire_project/data/train/fire
복사 완료: /kaggle/working/data/train/smoke → /content/wildfire_project/data/train/smoke
복사 완료: /kaggle/working/data/val/fire → /content/wildfire_project/data/val/fire
복사 완료: /kaggle/working/data/val/smoke → /content/wildfire_project/data/val/smoke
복사 완료: /kaggle/working/data/test/fire → /content/wildfire_project/data/test/fire
복사 완료: /kaggle/working/data/test/smoke → /content/wildfire_project/data/test/smoke


In [ ]:
## 파일 개수 합으로 전처리 확인하기

import os

base_dir = "/content/wildfire_project/data"
splits = ["train", "val", "test"]
classes = ["fire", "normal", "smoke"]

# 클래스별 전체 개수 초기화
total_counts = {cls: 0 for cls in classes}

print("각 폴더별 파일 개수:")

for split in splits:
    for cls in classes:
        folder = os.path.join(base_dir, split, cls)
        if os.path.exists(folder):
            files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
            count = len(files)
            total_counts[cls] += count
            print(f"{folder}: {count}개")
        else:
            print(f"{folder}: 경로 없음")

print("\n클래스별 총 파일 개수:")
for cls, count in total_counts.items():
    print(f"{cls}: {count}개")


각 폴더별 파일 개수:
/content/wildfire_project/data/train/fire: 4436개
/content/wildfire_project/data/train/normal: 608개
/content/wildfire_project/data/train/smoke: 3836개
/content/wildfire_project/data/val/fire: 1031개
/content/wildfire_project/data/val/normal: 152개
/content/wildfire_project/data/val/smoke: 845개
/content/wildfire_project/data/test/fire: 1115개
/content/wildfire_project/data/test/normal: 0개
/content/wildfire_project/data/test/smoke: 1186개

클래스별 총 파일 개수:
fire: 6582개
normal: 760개
smoke: 5867개


In [ ]:

# kaggle/input/..의 Testing 파일들에서는 fire, nofire이 분류되어 있지 않고
# 파일 개수도 몇 개씩인지 파악되지 않음.
# 위 코드까지는 각각의 파일 개수가 파악이 되므로 파일 수 합산으로 정상분류 확인
# kaggle/input/..의 Testing 디렉토리에서 파일 이름을 기준으로 content/.../test/fire 또는 nofire로 분류+복사하기

In [ ]:
import os
import shutil

src_dir = "/kaggle/input/forest-fire-dataset/Forest Fire Dataset/Testing"
dst_fire = "/content/wildfire_project/data/test/fire"
dst_nofire = "/content/wildfire_project/data/test/normal"

# 대상 디렉토리 생성
os.makedirs(dst_fire, exist_ok=True)
os.makedirs(dst_nofire, exist_ok=True)

# 파일 복사
for filename in os.listdir(src_dir):
    if not filename.lower().endswith((".jpg")):
        continue  # 이미지 파일만 처리

    lower_name = filename.lower()
    src_path = os.path.join(src_dir, filename)

    if "nofire" in lower_name:
        dst_path = os.path.join(dst_nofire, filename)
        shutil.copy(src_path, dst_path)
    elif "fire" in lower_name:
        dst_path = os.path.join(dst_fire, filename)
        shutil.copy(src_path, dst_path)
    else:
        print(f"분류 불가 파일: {filename}")

print("복사 완료")


복사 완료


In [ ]:
# ## 코드 재실행으로 인해 content/... 아래 내용 모두 삭제........
# import shutil
# import os

# content_path = "/content"

# if os.path.exists(content_path):
#     shutil.rmtree(content_path)
#     print(f"{content_path} 디렉토리와 그 내부 모든 내용을 삭제했습니다.")
# else:
#     print(f"{content_path} 경로가 존재하지 않습니다.")


/content 디렉토리와 그 내부 모든 내용을 삭제했습니다.


In [ ]:
## 다시 파일 개수 합 확인하기 - fire, nofire TESTING 디렉토리 확인용
# fire 총 합으로 정상적으로 분류된지는 알 수 없음.
# 다만 normale에 파일이 정상적으로 들어왔는지만 확인

import os

base_dir = "/content/wildfire_project/data"
splits = ["train", "val", "test"]
classes = ["fire", "normal", "smoke"]

# 클래스별 전체 개수 초기화
total_counts = {cls: 0 for cls in classes}

print("각 폴더별 파일 개수:")

for split in splits:
    for cls in classes:
        folder = os.path.join(base_dir, split, cls)
        if os.path.exists(folder):
            files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
            count = len(files)
            total_counts[cls] += count
            print(f"{folder}: {count}개")
        else:
            print(f"{folder}: 경로 없음")

print("\n클래스별 총 파일 개수:")
for cls, count in total_counts.items():
    print(f"{cls}: {count}개")


각 폴더별 파일 개수:
/content/wildfire_project/data/train/fire: 4436개
/content/wildfire_project/data/train/normal: 608개
/content/wildfire_project/data/train/smoke: 3836개
/content/wildfire_project/data/val/fire: 1031개
/content/wildfire_project/data/val/normal: 152개
/content/wildfire_project/data/val/smoke: 845개
/content/wildfire_project/data/test/fire: 1305개
/content/wildfire_project/data/test/normal: 190개
/content/wildfire_project/data/test/smoke: 1186개

클래스별 총 파일 개수:
fire: 6772개
normal: 950개
smoke: 5867개


In [ ]:
# ## 복사 완료 했으니 kaggle/working/... 내용 삭제하기
# import shutil
# import os

# # 삭제할 경로
# target_dir = "/kaggle/working"

# # 경로가 존재할 때만 삭제
# if os.path.exists(target_dir):
#     shutil.rmtree(target_dir)
#     print(f"'{target_dir}' 디렉토리와 하위 모든 파일/폴더를 삭제했습니다.")
# else:
#     print(f"'{target_dir}' 디렉토리가 존재하지 않습니다.")


In [ ]:
## 여기까지 파일 분류 완료!

In [ ]:
import torch  # new
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 하이퍼파라미터
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
NUM_CLASSES = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 경로 설정
DATA_DIR = "/content/wildfire_project/data"
TRAIN_DIR = f"{DATA_DIR}/train"
VAL_DIR = f"{DATA_DIR}/val"
TEST_DIR = f"{DATA_DIR}/test"

# 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# 데이터셋 불러오기
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=transform)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=transform)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 클래스 이름
class_names = train_dataset.classes

# VGG 모델 설정
model = models.vgg16(pretrained=True)
model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
model = model.to(DEVICE)

# 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 학습
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}, Train Accuracy: {acc:.4f}")

# 테스트 데이터 예측
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in DataLoader(test_dataset, batch_size=32):
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=class_names)

print("\n[Classification Report]")
print(report)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:06<00:00, 80.0MB/s]


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np

# 클래스 이름
class_names = train_dataset.classes

# 예측 및 라벨 수집
y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# confusion matrix 생성
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=class_names)

print("\n[분류 보고서]")
print(report)

# confusion matrix 시각화
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("예측 클래스")
plt.ylabel("실제 클래스")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()
